## This notebook can be used to rank a list of nodes from a category that connect to an entity such as a gene. 
#### Example 1: Which disease are associate with NPM1 gene?
#### Example 2: Which proteins interacts with protein NPM1?
#### Example 3: Which drugs or small molecules can interact with NPM1?


In [ ]:
from TCT import translator_metakg, translator_query, TCT

In [ ]:
# Step 1: List all the APIs in the translator system
APInames_current = TCT.get_Translator_APIs()
print(f"Number of current APIs: {len(APInames_current)}")

In [ ]:
# Step 2: Load Translator resources (APIs, metaKG)
from TCT.translator_resources import TranslatorResources
resources = TranslatorResources.load()

print(f"metaKG columns: {list(resources.meta_kg.columns)}")
print(f"metaKG shape: {resources.meta_kg.shape}")

In [ ]:
# Step 3: set input parameters
# Test multiomics BigGIM Drug Response KP
# Node1 for query
input_node1 = 'NPM1'
input_node1_id = TCT.get_curie(name=input_node1)
print(input_node1_id)
input_node1_list = [input_node1_id]
input_node1_category = ['biolink:Gene'] # Node: this has to be in a format of biolink:xxx

#Node2 for query
input_node2_list = []
#input_node2_category = ['biolink:Drug', 'biolink:SmallMolecule', 'biolink:ChemicalSubstance']
input_node2_category = ['biolink:Gene']

# Get all predicates for the input node1 and node2, user can furter select the predicates among this list
sele_predicates = list(set(TCT.select_concept(sub_list=input_node1_category,
                                              obj_list=input_node2_category,
                                              metaKG=resources.meta_kg)))
print("all relevant predicates in Translator:")
print(sele_predicates)
# select predicates

# Get all APIs for the input node1 and node2, user can furter select the APIs among this list
sele_APIs = TCT.select_API(sub_list=input_node1_category,
                           obj_list=input_node2_category,
                           metaKG=resources.meta_kg)

print("all relevant APIs in Translator:")
print(sele_APIs)
print(len(sele_APIs))

# get API URLs
API_URLs = TCT.get_Translator_API_URL(API_sele=sele_APIs,
                                      APInames=resources.api_names)

In [ ]:
# Step 4: Format query json
query_json = TCT.format_query_json(subject_ids=input_node1_list,
                                   object_ids=input_node2_list,
                                   subject_categories=input_node1_category,
                                   object_categories=input_node2_category,
                                   predicates=sele_predicates)

# Step 5: Query Translator APIs and parse results
result = translator_query.parallel_api_query(query_json=query_json,
                                             select_APIs=list(sele_APIs),
                                             resources=resources,
                                             max_workers=len(sele_APIs))

# Step 6: Parse results
result_parsed = TCT.parse_KG(result=result)

# Step 7: Ranking the results
result_ranked_by_primary_infores = TCT.rank_by_primary_infores(result_parsed=result_parsed, input_node=input_node1_id)

In [ ]:
print(query_json)

In [ ]:
# Step 8: Visualize the results
TCT.visulization_one_hop_ranking(result_ranked_by_primary_infores=result_ranked_by_primary_infores,
                                result_parsed=result_parsed,
                                num_of_nodes=30,
                                input_query=input_node1_id,
                                fontsize=10)


In [ ]:
# End of the example